# Reference Experiment RT2

Two bounded modes: `PREPARE_CANDIDATES` creates the one-time AI review pack without model inference; `EVALUATE_BENCHMARK` evaluates a later AI-curated pseudo-GT benchmark. RT2 is not a production milestone or an official competition evaluation.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = os.environ.get("AIC_REPO_URL", "https://github.com/Irthn1311/AIC2026_TeamPTK_SGU.git")
REPO_REF = os.environ.get("AIC_REPO_REF", "TRIAGEEG")
REPO_DIR = Path("/kaggle/working/AIC2026_TeamPTK_SGU")
if not (REPO_DIR / ".git").is_dir():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)],
        check=True,
    )
sys.path.insert(0, str(REPO_DIR / "src"))
COMMIT = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, capture_output=True, text=True, check=True
).stdout.strip()
print("resolved commit:", COMMIT)

In [ ]:
RT2_MODE = os.environ.get("AIC_RT2_MODE", "PREPARE_CANDIDATES").upper()
STAGE1_INPUT = os.environ.get("AIC_STAGE1_ROOT", "/kaggle/input/datasets/irthn1311/triage-eg-stage1b-input-bundle")
DATASET_ROOT = Path(os.environ.get("AIC_DATA_ROOT", "/kaggle/input/datasets/nadkli/dataset-aic"))
STAGE1B_INPUT = os.environ.get("AIC_STAGE1B_ROOT", "/kaggle/input/datasets/irthn1311/triage-eg-stage1b-encoder-compatibility-reports")
STAGE1E_INPUT = os.environ.get("AIC_STAGE1E_ROOT", "/kaggle/input/datasets/irthn1311/triage-eg-stage1e-language-path-freeze")
CLIP_INPUT = os.environ.get("AIC_CLIP_ROOT", "/kaggle/input/datasets/irthn1311/aic2026-openai-clip-vit-b32")
OPUS_INPUT = os.environ.get("AIC_OPUS_ROOT", "/kaggle/input/datasets/irthn1311/aic2026-opus-mt-vi-en")
BENCHMARK_INPUT = os.environ.get("AIC_RT2_BENCHMARK_ROOT", "/kaggle/input/datasets/irthn1311/triage-eg-rt2-ai-benchmark")
CANDIDATE_OUTPUT = Path("/kaggle/working/triage_eg_rt2_benchmark_candidates")
EVALUATION_OUTPUT = Path("/kaggle/working/triage_eg_rt2_evaluation")
print({"mode": RT2_MODE, "stage1": STAGE1_INPUT, "dataset": str(DATASET_ROOT)})

In [ ]:
def resolve_root(root, marker, max_depth=3):
    root = Path(root)
    matches, frontier = [], [(root, 0)]
    while frontier:
        current, depth = frontier.pop(0)
        if (current / marker).is_file():
            matches.append(current)
            continue
        if depth < max_depth and current.is_dir():
            frontier.extend((child, depth + 1) for child in sorted(current.iterdir()) if child.is_dir())
    if len(matches) != 1:
        raise RuntimeError(f"Expected one root containing {marker} below {root}; found {matches}")
    return matches[0]

if RT2_MODE not in {"PREPARE_CANDIDATES", "EVALUATE_BENCHMARK"}:
    raise ValueError(f"Unsupported RT2 mode: {RT2_MODE}")
STAGE1_ROOT = resolve_root(STAGE1_INPUT, "stage1_summary.json")
if not DATASET_ROOT.is_dir():
    raise RuntimeError(f"Missing dataset root: {DATASET_ROOT}")
if RT2_MODE == "EVALUATE_BENCHMARK":
    STAGE1B_ROOT = resolve_root(STAGE1B_INPUT, "stage1b_summary.json")
    STAGE1E_ROOT = resolve_root(STAGE1E_INPUT, "language_path_contract.json")
    CLIP_ROOT = resolve_root(CLIP_INPUT, "manifests/asset_manifest.json")
    OPUS_ROOT = resolve_root(OPUS_INPUT, "manifests/asset_manifest.json")
    BENCHMARK_PATH = resolve_root(BENCHMARK_INPUT, "rt2_ai_benchmark.jsonl") / "rt2_ai_benchmark.jsonl"
print("resolved Stage 1 root:", STAGE1_ROOT)

In [ ]:
from triage_eg.experiments.reference_rt2 import load_rt2_settings

SETTINGS = load_rt2_settings(REPO_DIR / "configs/experiments/reference_rt2.yaml")
print({"seed": SETTINGS.seed, "lambda_grid": SETTINGS.lambda_grid})

In [ ]:
if RT2_MODE == "PREPARE_CANDIDATES":
    from triage_eg.experiments.reference_rt2 import create_candidate_bundle, prepare_benchmark_candidates

    RESULT = prepare_benchmark_candidates(
        STAGE1_ROOT, DATASET_ROOT, CANDIDATE_OUTPUT,
        candidate_count=SETTINGS.candidate_count, seed=SETTINGS.seed,
        frames_per_sheet=SETTINGS.frames_per_sheet, build_git_commit=COMMIT,
    )
    ZIP_PATH = Path("/kaggle/working/triage_eg_rt2_benchmark_candidates.zip")
    create_candidate_bundle(CANDIDATE_OUTPUT, ZIP_PATH)
    print(json.dumps({k: RESULT[k] for k in ("eligible_video_count", "selected_video_count", "bucket_counts")}, indent=2))

In [ ]:
if RT2_MODE == "EVALUATE_BENCHMARK":
    from triage_eg.experiments.reference_rt2 import (
        RT2RunnerConfig, create_rt2_evaluation_bundle, load_rt2_benchmark,
        run_reference_rt2_evaluation,
    )
    from triage_eg.retrieval.stage2 import config_from_yaml

    stage2 = config_from_yaml(
        REPO_DIR / "configs/retrieval/stage2_operational_runtime.yaml",
        stage1_root=STAGE1_ROOT, stage1b_root=STAGE1B_ROOT, stage1e_root=STAGE1E_ROOT,
        clip_asset_root=CLIP_ROOT, translator_asset_root=OPUS_ROOT,
        output_root=EVALUATION_OUTPUT / "_stage2_control",
        stage1d_config=REPO_DIR / "configs/retrieval/stage1d_translation_ablation.yaml",
        build_git_commit=COMMIT,
    )
    QUERIES = load_rt2_benchmark(BENCHMARK_PATH)
    RESULT = run_reference_rt2_evaluation(
        RT2RunnerConfig(stage2, DATASET_ROOT, BENCHMARK_PATH, EVALUATION_OUTPUT, SETTINGS),
        QUERIES,
    )
    ZIP_PATH = Path("/kaggle/working/triage_eg_rt2_evaluation_bundle.zip")
    create_rt2_evaluation_bundle(EVALUATION_OUTPUT, ZIP_PATH)
    print(json.dumps(RESULT, indent=2))

In [ ]:
from IPython.display import Image, display

if RT2_MODE == "PREPARE_CANDIDATES":
    for path in sorted((CANDIDATE_OUTPUT / "candidates").glob("*.jpg"))[:4]:
        print(path.stem)
        display(Image(filename=str(path)))
elif (EVALUATION_OUTPUT / "visuals").is_dir():
    for path in sorted((EVALUATION_OUTPUT / "visuals").glob("*_ab.jpg")):
        print(path.stem)
        display(Image(filename=str(path)))

In [ ]:
print("DOWNLOAD ZIP:", ZIP_PATH, "size_bytes=", ZIP_PATH.stat().st_size)
if RT2_MODE == "PREPARE_CANDIDATES":
    print("RT2_IMPLEMENTATION_STATUS = COMPLETE")
    print("RT2_CANDIDATE_PACK_STATUS = READY")
    print("RT2_BENCHMARK_STATUS = WAITING_FOR_AI_CURATED_LABELS")
    print("DANTE_QUALITY_DECISION = NOT_EVALUATED")
else:
    print("RT2_BENCHMARK_STATUS =", RESULT["calibration_status"])
    print("DANTE_QUALITY_DECISION = NOT_EVALUATED")